In [1]:
#!pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org scikit-posthocs

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from scipy import stats
import matplotlib.ticker as mticker

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Base project directory
BASE_PATH = '..' # Currently active path

# Input Files
SCENARIO_FILE = 'consumo_previsto_todos_cenarios_2026_2035.csv'
SARIMA_FILE = 'previsao_consumo_sarima_2026_2035.csv'

# Output File
OUTPUT_FIG = 'normality_analysis_scenarios_vs_sarima.pdf'

# Directory Setup
INPUT_DIR_RES = os.path.join(BASE_PATH, 'resultados')
INPUT_DIR_MODELS = os.path.join(BASE_PATH, 'modelos IA')
INPUT_DIR_INC = os.path.join(BASE_PATH, 'includes')
OUTPUT_DIR = os.path.join(BASE_PATH, 'figuras')

# Style configuration
try:
    plt.style.use('seaborn-v0_8-white')
except:
    try:
        plt.style.use('seaborn-white')
    except:
        plt.style.use('default')

sns.set_palette("husl")

def sum_yearly(df, date_col, value_cols):
    """ Helper function to aggregate monthly data into annual sums """
    df_year = df.copy()
    df_year['Year'] = df_year[date_col].dt.year
    return df_year.groupby('Year')[value_cols].sum().reset_index()

def main():
    print("--- Starting Normality Analysis: 18 Scenarios vs SARIMA ---")

    # ==========================================
    # 2. Load and Prepare Data
    # ==========================================
    
    # Load Scenario Data (18 scenarios)
    path_scenarios = os.path.join(INPUT_DIR_RES, SCENARIO_FILE)
    if not os.path.exists(path_scenarios):
        path_scenarios = os.path.join(INPUT_DIR_INC, SCENARIO_FILE)
    
    # Using semicolon separator and comma for decimals as per the 18-scenario file format
    df_scenarios = pd.read_csv(path_scenarios, sep=';', decimal=',')
    
    # Date Construction
    if 'Mes' in df_scenarios.columns and 'Ano' in df_scenarios.columns:
        df_scenarios['Date'] = pd.to_datetime(df_scenarios['Ano'].astype(str) + '-' + 
                                            df_scenarios['Mes'].astype(str) + '-01')
    elif 'Mes_Ano' in df_scenarios.columns:
        df_scenarios['Date'] = pd.to_datetime(df_scenarios['Mes_Ano'], format='%m/%Y')
    
    # Identify Scenario Columns
    metadata_cols = ['Mes', 'Ano', 'Mes_Ano', 'Date']
    scenario_cols = [c for c in df_scenarios.columns if c not in metadata_cols]
    df_scenarios = df_scenarios[df_scenarios['Date'] <= pd.to_datetime('2035-12-31')]

    # Load SARIMA Data
    path_sarima = os.path.join(INPUT_DIR_MODELS, SARIMA_FILE)
    if not os.path.exists(path_sarima):
        print(f"Error: SARIMA file not found at {path_sarima}")
        return
    
    df_sarima = pd.read_csv(path_sarima)
    df_sarima = df_sarima.rename(columns={'Date': 'Date', 'Projected_Consumption': 'SARIMA'})
    df_sarima['Date'] = pd.to_datetime(df_sarima['Date'])
    df_sarima = df_sarima[df_sarima['Date'] <= pd.to_datetime('2035-12-31')]
    
    print(f"SARIMA data loaded: {len(df_sarima)} months")

    # Merge and Aggregate
    df_combined = pd.merge(df_scenarios, df_sarima[['Date', 'SARIMA']], on='Date', how='inner')
    all_series = scenario_cols + ['SARIMA']
    df_annual = sum_yearly(df_combined, 'Date', all_series)
    
    print(f"Annual data shape: {df_annual.shape}")
    print(f"Years analyzed: {df_annual['Year'].min()} - {df_annual['Year'].max()}")

    # ==========================================
    # 3. Graphical Analysis (Distribution & Q-Q Plots)
    # ==========================================
    
    # Save all plots in a single PDF with multiple pages
    from matplotlib.backends.backend_pdf import PdfPages
    
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FIG)
    
    with PdfPages(output_path) as pdf:
        
        # --- Page 1: Histograms / KDE ---
        plt.figure(figsize=(14, 8))
        for col in all_series:
            # Use Scenario IDs for labels
            label_id = col.split('_')[0] if '_' in col else col
            sns.histplot(df_annual[col], kde=True, label=label_id, element="step", alpha=0.1, bins=10)
            
        plt.title('Distribution Comparison: 18 Scenarios and SARIMA Model', fontsize=16, fontweight='bold')
        plt.xlabel('Annual Consumption (m³)', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=1)
        plt.tight_layout()
        pdf.savefig()
        plt.close()

        # --- Page 2 & 3: Q-Q Plots (Grids) ---
        # Splitting into two pages to maintain readability (19 total series: 18 scenarios + SARIMA)
        series_chunks = [all_series[i:i + 10] for i in range(0, len(all_series), 10)]
        
        for chunk in series_chunks:
            n_plots = len(chunk)
            n_rows = (n_plots + 2) // 3  # Ceiling division to get appropriate rows
            n_rows = max(3, n_rows)  # Minimum 3 rows for layout
            n_cols = 3
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
            axes = axes.flatten() if n_rows > 1 else [axes]
            
            for i, scenario in enumerate(chunk):
                label_id = scenario.split('_')[0] if '_' in scenario else scenario
                stats.probplot(df_annual[scenario], plot=axes[i])
                axes[i].set_title(f'Q-Q Plot: {label_id}', fontsize=10)
                axes[i].grid(True, alpha=0.3)
            
            # Hide unused subplots
            for j in range(len(chunk), len(axes)):
                axes[j].set_visible(False)
                
            plt.tight_layout()
            pdf.savefig()
            plt.close()

    # ==========================================
    # 4. Statistical Normality Tests (Shapiro-Wilk)
    # ==========================================
    
    normality_results = []
    for scenario in all_series:
        label_id = scenario.split('_')[0] if '_' in scenario else scenario
        try:
            stat, p = stats.shapiro(df_annual[scenario])
            normality_results.append({
                'Model/Scenario': label_id,
                'Statistic (W)': stat,
                'p-value': p,
                'Normal (p>0.05)': 'Yes' if p > 0.05 else 'No'
            })
        except Exception as e:
            print(f"Error processing {label_id}: {e}")
            normality_results.append({
                'Model/Scenario': label_id,
                'Statistic (W)': np.nan,
                'p-value': np.nan,
                'Normal (p>0.05)': 'Error'
            })

    normality_df = pd.DataFrame(normality_results)
    
    print("\n" + "="*70)
    print("Shapiro-Wilk Normality Test Results")
    print("="*70)
    print(normality_df.round(4).to_string(index=False))
    print("="*70)
    
    # Summary statistics
    normal_count = len(normality_df[normality_df['Normal (p>0.05)'] == 'Yes'])
    non_normal_count = len(normality_df[normality_df['Normal (p>0.05)'] == 'No'])
    
    print(f"\nSummary:")
    print(f"  - Normal distributions: {normal_count}")
    print(f"  - Non-normal distributions: {non_normal_count}")
    
    # Check SARIMA specifically
    sarima_row = normality_df[normality_df['Model/Scenario'] == 'SARIMA']
    if not sarima_row.empty:
        p_value = sarima_row['p-value'].values[0]
        print(f"\nSARIMA p-value: {p_value:.4f}")
        if p_value < 0.05:
            print("  → SARIMA rejects normality (p < 0.05)")
        else:
            print("  → SARIMA does not reject normality (p > 0.05)")
    
    # Save the table results to a CSV for documentation
    table_output = os.path.join(INPUT_DIR_RES, 'normality_test_results_sarima_2026_2035.csv')
    normality_df.to_csv(table_output, index=False, sep=';', decimal=',')
    print(f"\nResults saved to: {table_output}")
    print(f"Plot saved to: {output_path}")
    print("\n--- Analysis Complete ---")

if __name__ == "__main__":
    main()

--- Starting Normality Analysis: 18 Scenarios vs SARIMA ---
SARIMA data loaded: 120 months
Annual data shape: (10, 20)
Years analyzed: 2026 - 2035



Shapiro-Wilk Normality Test Results
Model/Scenario  Statistic (W)  p-value Normal (p>0.05)
            CI         0.9173   0.3347             Yes
           CII         0.9173   0.3352             Yes
          CIII         0.9172   0.3343             Yes
           CIV         0.9701   0.8914             Yes
            CV         0.9702   0.8927             Yes
           CVI         0.9703   0.8932             Yes
          CVII         0.9701   0.8916             Yes
         CVIII         0.9702   0.8928             Yes
           CIX         0.9703   0.8933             Yes
            CX         0.9700   0.8908             Yes
           CXI         0.9701   0.8920             Yes
          CXII         0.9701   0.8920             Yes
         CXIII         0.9701   0.8917             Yes
          CXIV         0.9702   0.8930             Yes
           CXV         0.9703   0.8937             Yes
          CXVI         0.9046   0.2459             Yes
         CXVII         0.897